In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import re
from pathlib import Path
from datetime import date, timedelta
from natsort import natsort_keygen, natsorted, ns
from xlsxwriter.utility import xl_col_to_name
from datetime import datetime

import sys
sys.path.insert(0, str(Path("../src").resolve()))
import ado_reporting as cl
import daily_charge as dc
import work_items_report as wir
import data_validation as dv
import excel_import as xi
import burndown as bd

pd.options.display.max_columns = None  # type: ignore
pd.options.display.float_format = "{:,.2f}".format


In [2]:
# ==== USER CONFIG ====
project = "MarketRMS"

start_date = date(2026, 3, 1)  # <- report generated since that date

# today = date.today()

today = "2026-04-01"
today = datetime.strptime(today, "%Y-%m-%d").date()

# --- Validate 'today' ---
today = dv.validate_today(today)

DAYS_BACK = (today - start_date).days + 1

# Paths
BASE_DIR = Path("..") / "data"

# Categories
leaf_types = ["Task", "Bug"]
removed_state = "Removed"
doneish_statuses = {"Code Review", "Ready for UAT", "Ready for Prod", "In UAT", "Closed"}
daily_charge_output_path = BASE_DIR / f"{project}_daily_charge_report.xlsx"
today_ts = pd.Timestamp(today)
today_str = today.strftime("%Y-%m-%d")
output_file = BASE_DIR / f"{project}_ProjectStatus_{today_str}_ADO_API.xlsx"
output_file_2 = BASE_DIR / f"{project}_Utilisation_{today_str}.xlsx"
burndown_file = BASE_DIR / f"{project}_Burndown_{today_str}.xlsx"


# Prepare shapshots dataset for analysis

In [3]:
wi_raw_snapshots = cl.get_wi_snapshots(
    project=project,
    days_back=DAYS_BACK,
    removed_state=removed_state,
)

# --- Validate source data freshness ---
_latest_snap = pd.Timestamp(wi_raw_snapshots["SnapshotDate"].max()).date()
if _latest_snap < today - timedelta(days=1):
    raise RuntimeError(
        "Project snapshots data is outdated. Please run synchronisation with ADO."
    )


Historical data selected: 2026-02-28 - 2026-03-31
Latest live snapshot file: MarketRMS_live_snapshot_2026-04-01.parquet (date=2026-04-01)
Live rows loaded: 1923
Final dataset date range: 2026-02-28 → 2026-04-01


# Identify related sprints

In [4]:
sprints, current_sprint, previous_sprint = cl.get_sprint_info(wi_raw_snapshots, today)

previous_sprint, current_sprint

('Sprint 35 (16.03-27.03)', 'Sprint 36 (30.03-10.04)')

# Charge by day report

In [5]:
# Compute daily charge tables (per assignee, per task) from snapshots.
# See src/daily_charge.py for the calculation.


In [6]:
# --- Daily charge by Assignee ---
daily_charge_by_person = dc.compute_daily_charge_by_person(wi_raw_snapshots)


In [7]:
# --- Daily charge by Task ---
daily_charge_by_task = dc.compute_daily_charge_by_task(wi_raw_snapshots)


In [8]:
# --- Download Daily charge for Power BI ---
daily_charge_by_person_bi = dc.export_daily_charge_for_powerbi(
    daily_charge_by_person,
    BASE_DIR / f"{project}_PowerBI_daily_charge.csv",
)


# Report for Daily (Stand-Ups)

In [9]:
# Build the daily meeting (stand-up) report.
# See src/work_items_report.py for the calculation.
daily_meeting_filtered, completed_cols = wir.compute_daily_meeting(
    wi_raw_snapshots, daily_charge_by_task, today
)
col_completed_earliest, col_completed_mid, col_completed_latest = completed_cols


# Sprint Burndown

In [ ]:
# Compute sprint burndowns (overall and per-assignee) from snapshots.
# See src/burndown.py for the calculation.
bd_current = bd.compute_sprint_burndown(wi_raw_snapshots, current_sprint)
bd_prev = bd.compute_sprint_burndown(wi_raw_snapshots, previous_sprint)


In [ ]:
# Per-assignee burndown for the current sprint (wide pivot for Excel).
bd_assignee_pivot = bd.compute_burndown_by_assignee(wi_raw_snapshots, current_sprint)


In [13]:
# Save the Team Analytics Dashboard workbook.
# See src/excel_import.py for the per-sheet writers and formatting rules.
xi.save_team_analytics_report(
    output_file,
    summary=daily_meeting_filtered,
    completed_cols=completed_cols,
    daily_charge_by_person=daily_charge_by_person,
    daily_charge_by_task=daily_charge_by_task,
    bd_current=bd_current,
    bd_prev=bd_prev,
    bd_assignee_pivot=bd_assignee_pivot,
    current_sprint=current_sprint,
    previous_sprint=previous_sprint,
)

print(f"\u2705 Done! File saved: {output_file}")


✅ Done! File saved: ../data/MarketRMS_ProjectStatus_2026-04-01_ADO_API.xlsx


## 2. Utilisation calculation

In [14]:
# Prepare data for calculation

sprints = sprints[
    (~sprints['IterationStart'].isna()) &
    (~sprints['IterationEnd'].isna())
]
sprints['IterationStart'] = pd.to_datetime(sprints["IterationStart"])
sprints['IterationEnd'] = pd.to_datetime(sprints["IterationEnd"])

# Filter source whole dataset for utilisation calculation
utilisation = wi_raw_snapshots[["SnapshotDate", "IterationName", "IterationStart", "IterationEnd", "WorkItemType", "Title", "State", "AssignedTo", "CompletedWork", "OriginalEstimate", "RemainingWork", "WorkItemURL"]].copy()
utilisation['IterationStart'] = pd.to_datetime(utilisation["IterationStart"])
utilisation['IterationEnd'] = pd.to_datetime(utilisation["IterationEnd"])
utilisation = utilisation[utilisation["IterationStart"] >= pd.to_datetime(start_date)]

# Select Snapshots at the first day of each Sprint (it should be day's night, so Snapshot is after planning)
utilisation = utilisation[utilisation["SnapshotDate"] == utilisation["IterationStart"]]

# Calculate CompletedWork at the start of sprint
utilisation["PrevDate"] = utilisation["SnapshotDate"] - pd.Timedelta(days=1)
utilisation = utilisation.merge(
    wi_raw_snapshots[["SnapshotDate", "WorkItemURL", "CompletedWork"]],
    left_on=["WorkItemURL", "PrevDate"],
    right_on=["WorkItemURL", "SnapshotDate"],
    how="left",
    suffixes=('', '_start')
).drop(columns = ["SnapshotDate_start", "PrevDate"])
utilisation["Completed_at_sprint_start"] = utilisation["CompletedWork_start"].fillna(0)
utilisation = utilisation.drop(columns = ["CompletedWork", "CompletedWork_start"])

# Calculate Planned_for_sprint
utilisation["Planned_for_sprint"] = (utilisation["OriginalEstimate"] - utilisation["Completed_at_sprint_start"]).clip(lower=0)

### 2-1 Charged time by sprints

In [15]:
# Charged time by sprints

# 1 Melt table with charges
id_cols = ["AssignedTo", "IterationName", "WorkItemType", "Title", "WorkItemURL"]
date_cols = [c for c in daily_charge_by_task.columns if c not in id_cols + ["total_charge"]]
charge_by_sprint = daily_charge_by_task.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="SnapshotDate",
    value_name="charge"
)

charge_by_sprint["SnapshotDate"] = pd.to_datetime(charge_by_sprint["SnapshotDate"], errors="coerce")
charge_by_sprint = charge_by_sprint[charge_by_sprint["charge"].fillna(0) != 0]


# Add 'IterationStart', 'IterationEnd' to Iteration name
charge_by_sprint = charge_by_sprint.merge(
    sprints[['IterationName', 'IterationStart', 'IterationEnd']],
    how='left',
    left_on=None,
    right_on=None,
    suffixes=('', '_s')
)

# 2 Match Iteration for each charged date - WorkingIteration
intervals = pd.IntervalIndex.from_arrays(sprints['IterationStart'], sprints['IterationEnd'], closed='both')
idx = intervals.get_indexer(charge_by_sprint['SnapshotDate'])
charge_by_sprint['WorkingIteration'] = np.where(idx >= 0, sprints['IterationName'].to_numpy()[idx], pd.NA)

# 3 Group daily disagregation to Iteration disagregation
charge_by_sprint = (
    charge_by_sprint.groupby(['WorkingIteration', 'AssignedTo', 'WorkItemType', 'Title', 'WorkItemURL'], as_index=False)['charge']
        .sum()
)
charge_by_sprint = charge_by_sprint.dropna(subset=['WorkingIteration'])

### 2-2 For each spring - split planned time and unplanned time

In [16]:
# Merge planned time charges to each sprtint tasks 

utilisation = utilisation.merge(
    charge_by_sprint[["WorkItemURL", "WorkingIteration", "charge"]],
    left_on = ["WorkItemURL", "IterationName"],
    right_on = ["WorkItemURL", "WorkingIteration"],
    how = "left"
)
utilisation["charge"] = utilisation["charge"].fillna(0)

# Mark them as planned
utilisation["Planned"] = "yes"

# Formatting
utilisation = utilisation[["IterationName", "AssignedTo", "WorkItemType", "Title", "Planned", "Planned_for_sprint", "charge", "WorkItemURL"]]


In [17]:
# Filter charges from 'charge_by_sprint', which were not planned for each sprint (utilisation["WorkItemURL"] does not match with charge_by_sprint["WorkingIteration"]) 
not_planned_charge = charge_by_sprint.merge(
    utilisation[["WorkItemURL", "IterationName"]],
    left_on = ["WorkItemURL", "WorkingIteration"],
    right_on = ["WorkItemURL", "IterationName"],
    how = "left"
)
not_planned_charge = not_planned_charge[not_planned_charge["IterationName"].isna()]

# Mark these as not planned
not_planned_charge["Planned"] = "no"

# Formatting
not_planned_charge["Planned_for_sprint"] = 0
not_planned_charge = not_planned_charge[["WorkingIteration", "AssignedTo", "WorkItemType", "Title", "Planned_for_sprint", "charge", "WorkItemURL", "Planned"]].rename(columns = {"AssignedTo": "AssignedTo", "WorkingIteration": "IterationName"})

# Combine planned tasks with not planned tasks - to single dataframe
utilisation = pd.concat([utilisation, not_planned_charge])

# Formatting
utilisation["Completed time"] = utilisation["charge"]


### 2-3 Calculate metrics

In [18]:
# Remove QA from this analysis
utilisation = utilisation[utilisation["AssignedTo"] != "Horin Serhii"]

# Select completed as planned to separate column
utilisation["Completed for planned tasks"] = np.where(
    utilisation["Planned"] == "yes",
    utilisation[["Completed time", "Planned_for_sprint"]].min(axis=1),
    0
)

# Account for underestimation
utilisation["Overcharged_planned_tasks"] = np.where(
    utilisation["Planned"] == "yes",
    (utilisation["Completed time"] - utilisation["Planned_for_sprint"]).clip(lower=0),
    0
)

# Calculate what time team failed to complete within planned sprint
utilisation["NOT DONE but planed"] = (utilisation["Planned_for_sprint"] - utilisation["Completed time"]).clip(lower=0)


# Select other completed time, which was not planned to be part of the sprint
utilisation["Completed non-sprint tasks"] = np.where(
    utilisation["Planned"] == "no",
    utilisation["Completed time"],
    0
)


In [19]:
utilisation_summary = (
    utilisation.groupby(['AssignedTo', 'IterationName'], as_index=False)
    [['Planned_for_sprint', 'Completed for planned tasks', 'NOT DONE but planed', 'Completed time', 'Completed non-sprint tasks', 'Overcharged_planned_tasks']].sum()
)

utilisation_time_map = {
    'Vasilov Denys':  60,        # 80 hours * 70%
    'Vasilov Denys S.': 60,      # 80 hours * 70%
    'Sukonnik Illia V.': 50,     # 80 hours * 60%
    'Kolodieiev Ivan': 50,       # 80 hours * 60%
    'Andrianov Vladyslav': 40,   # 80 hours * 50%
}

utilisation_summary['Utilisation time'] = (
    utilisation_summary['AssignedTo']
        .map(utilisation_time_map)
        .fillna(0)          # or raise / keep NaN
        .astype(int)
)

utilisation_summary = utilisation_summary.sort_values(["AssignedTo", "IterationName"], ascending=False)

utilisation_summary["Underutilised"] = utilisation_summary['Utilisation time'] - utilisation_summary["Completed time"]
utilisation_summary["Overplanned / Underplanned"] = utilisation_summary['Planned_for_sprint'] - utilisation_summary["Utilisation time"]
utilisation_summary["Work hours debt"] = np.maximum(np.minimum(utilisation_summary["Planned_for_sprint"], utilisation_summary["Utilisation time"]), utilisation_summary["Utilisation time"]) - utilisation_summary["Completed time"]
utilisation_summary["Comment on Plan Focus"] = ""
utilisation_summary["Comment on Utilisation"] = ""

utilisation_summary = utilisation_summary[["AssignedTo", "IterationName", "Utilisation time", "Planned_for_sprint", "Overplanned / Underplanned", "Completed for planned tasks", 'Overcharged_planned_tasks', "Completed non-sprint tasks", "Work hours debt", "Comment on Plan Focus"]]
# "Completed time", "Underutilised", "Comment on Utilisation",
# "NOT DONE but planed"

In [20]:
with pd.ExcelWriter(output_file_2, engine="xlsxwriter") as writer:
    # --- write sheets ---
    utilisation_summary.to_excel(writer, sheet_name="Summary", index=False)
    utilisation.to_excel(writer, sheet_name="Details", index=False)

    workbook  = writer.book

    header_fmt = workbook.add_format({
        "bold": True,
        "text_wrap": True,
        "valign": "top",
        "align": "center",
        "border": 1,
    })

    def format_sheet(df: pd.DataFrame, sheet_name: str, header_format, extra_pad: int = 2,
                     min_width: int = 10, max_width: int = 60):
        ws = writer.sheets[sheet_name]

        # Apply header format
        for col_idx, col_name in enumerate(df.columns):
            ws.write(0, col_idx, col_name, header_format)

        # Set column widths (based on max of header length and sampled cell lengths)
        for col_idx, col_name in enumerate(df.columns):
            # header length
            header_len = len(str(col_name))

            # content length (sample a bit for speed; adjust if you want full scan)
            if len(df) > 0:
                sample = df.iloc[:500, col_idx].astype(str)  # sample first 500 rows
                content_len = sample.map(len).max()
            else:
                content_len = 0

            width = max(header_len, content_len) + extra_pad
            width = max(min_width, min(width, max_width))

            ws.set_column(col_idx, col_idx, width)

        # Freeze header row + enable autofilter
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, 0, len(df.columns) - 1)

    format_sheet(utilisation_summary, "Summary", header_fmt)
    format_sheet(utilisation, "Details", header_fmt)

output_file_2

PosixPath('../data/MarketRMS_Utilisation_2026-04-01.xlsx')